# Qwen3-0.6B × Triton split-KV (block_ptr) AttentionBackend

**block_ptr/vllm_split_v2** — `ptr/vllm_split_v2` 의 메모리 접근을 `tl.make_block_ptr` 표현으로 옮긴 변형.

동일한 split-KV 알고리즘 (4 커널: prefill / decode_simple / decode_partial / decode_reduce) 을 사용하지만 모든 Q/K/V/O 로드·스토어가 block_ptr 통과:

- **prefill**: K 의 virtual transpose (`order=(0,1)`) 로 `tl.trans` 제거 — 기존 block_ptr/vllm_split 와 동일
- **decode_simple**: 1-D Q block_ptr + 2-D K/V block_ptr + `tl.advance` — 기존 block_ptr/vllm_split 와 동일
- **decode_partial (NEW)**: K/V block_ptr 의 `offsets=(n_start * BLOCK_N, 0)` 로 KV chunk 시작점 정렬, chunk 경계는 `tl.where` 로 마스킹
- **decode_reduce (NEW)**: partial buffer 를 1-D / 2-D block_ptr 로 깔끔하게 read, log-sum-exp 결합

**범위 고지**
- KV_SPLITS 는 wrapper 가 자동 결정 + 항상 power-of-2 보장 (`tl.arange` 제약)
- `MY_TRITON_BACKEND_LOG=1` 시 첫 호출에 `MyTritonImpl[v2/block_ptr].forward fired ...` 로그 1회

## 준비물 & 설치

- CUDA GPU 필수
- Python >= 3.10
- vLLM **0.19.1 정확히** 권장

```bash
# 노트북 디렉토리에서
pip install -e .
```

**같은 venv 에 ptr/vllm_split_v2 또는 다른 stage 와 동시 설치 금지** — `py-modules` 이름이 같아 충돌.

In [ ]:
import torch, triton, vllm

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm:', vllm.__version__)
print('triton:', triton.__version__)

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
)

props = torch.cuda.get_device_properties(0)
print(f'SMs: {props.multi_processor_count}')
print(f'compute capability: {torch.cuda.get_device_capability(0)}')

## Qwen3-0.6B 구조 요약

| 항목 | 값 |
|---|---|
| hidden_size | 1024 |
| Q heads | 16 |
| KV heads | 8 (GQA 2:1) |
| head_dim | 128 |
| layers | 28 |

Decode grid = B × Hq = 16 block. SM 수에 비해 작아 split-KV 의 이득 영역.

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden:', cfg.hidden_size)
print('Q heads:', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers:', cfg.num_hidden_layers)

## block_ptr 차이 — split-KV 의 표현

**ptr 변형** vs **block_ptr 변형** (동일 알고리즘, 다른 메모리 접근):

| 위치 | ptr 변형 | block_ptr 변형 |
|---|---|---|
| Q load | `tl.load(Q + pid_bh * sb + offs_d * sd)` | 1-D `make_block_ptr` |
| K/V load (decode loop) | `tl.load(K + ... offs_n[:,None] * ...)` + 매 iter offset 재계산 | 2-D `make_block_ptr` + `tl.advance(K, (BLOCK_N, 0))` |
| KV chunk 시작 (split-KV) | `K + ...` 에 `kv_start` 더해 매번 계산 | `make_block_ptr(offsets=(n_start * BLOCK_N, 0))` 한 번 + `tl.advance` |
| chunk 경계 mask | `mask_n = (offs_n >= kv_start) & (offs_n < kv_end)` | **동일** (block_ptr 의 boundary_check 가 chunk 경계 모름) |
| partial buffer write | scalar `tl.store(PartialM + ...)` + 벡터 `tl.store(PartialAcc + offs_d * ...)` | 벡터는 1-D block_ptr, scalar 는 raw pointer 유지 |
| reduce kernel | `offs_k = tl.arange(0, KV_SPLITS)` 로 직접 인덱싱 | 1-D / 2-D block_ptr 로 partial 일괄 load |

수학과 grid 토폴로지는 동일. 검증: 두 변형의 smoke test 가 같은 max_abs_err 보임.

In [ ]:
from triton_attn import triton_attention_decode, _choose_kv_splits, _pow2_floor

print(triton_attention_decode.__doc__)
print()
B, Hq = 1, 16
print(f'KV_SPLITS heuristic for B={B}, Hq={Hq} (grid_bh = {B*Hq}):')
for S in (32, 256, 512, 1024, 4096, 16384):
    ns = _choose_kv_splits(B * Hq, S)
    print(f'  S={S:>6}  →  KV_SPLITS={ns}  (always power of 2)')

In [ ]:
# Smoke test: prefill + decode with split-KV (block_ptr) vs SDPA reference
import torch.nn.functional as F
from triton_attn import triton_attention_prefill, triton_attention_decode

B, Hq, Hkv, D = 1, 16, 8, 128

print('prefill (q_len == kv_len, causal, block_ptr)')
for S in (128, 1024):
    q = torch.randn(B, Hq, S, D, dtype=torch.float16, device='cuda')
    k = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
    v = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
    ref = F.scaled_dot_product_attention(
        q, k.repeat_interleave(Hq // Hkv, dim=1),
        v.repeat_interleave(Hq // Hkv, dim=1), is_causal=True,
    )
    ours = triton_attention_prefill(q, k, v)
    err = (ours.float() - ref.float()).abs().max().item()
    print(f'  S={S:<6} max_err = {err:.6f}  →  {"PASS" if err < 1e-2 else "FAIL"}')

print()
print('decode (q_len=1, split-KV, block_ptr) — sweep KV_SPLITS')
for S in (128, 1024, 4096):
    q_full = torch.randn(B, Hq, S, D, dtype=torch.float16, device='cuda')
    k = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
    v = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
    ref_full = F.scaled_dot_product_attention(
        q_full, k.repeat_interleave(Hq // Hkv, dim=1),
        v.repeat_interleave(Hq // Hkv, dim=1), is_causal=True,
    )
    ref_last = ref_full[:, :, -1:, :]
    q_last = q_full[:, :, -1:, :].contiguous()
    print(f'  S={S}:')
    for ns in (None, 1, 2, 4, 8):
        if ns is not None and ns > S:
            continue
        ours = triton_attention_decode(q_last, k, v, kv_splits=ns)
        err = (ours.float() - ref_last.float()).abs().max().item()
        tag = 'auto' if ns is None else f'ns={ns}'
        print(f'    {tag:<6} max_err = {err:.6f}  →  {"PASS" if err < 1e-2 else "FAIL"}')

## Plugin entry point

`pyproject.toml`:
```toml
[project.entry-points."vllm.general_plugins"]
my_block_ptr_split_v2_backend = "triton_attention_backend:register"
```

실행 증거는 엔진 코어 stderr 의 `MyTritonImpl[v2/block_ptr].forward fired ...` 로그 — `[v2/block_ptr]` 태그가 ptr 변형 (`[v2]`) 와 구분.

In [ ]:
from importlib.metadata import entry_points

eps = list(entry_points(group='vllm.general_plugins'))
mine = [e for e in eps if e.name == 'my_block_ptr_split_v2_backend']
assert mine, (
    '`my_block_ptr_split_v2_backend` entry point 가 보이지 않는다. '
    '`pip install -e .` 을 실행했는지 확인하라.'
)
print('plugin registered:', mine[0].name, '->', mine[0].value)

In [ ]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.CUSTOM,
    enforce_eager=True,
    max_num_seqs=1,
    max_model_len=2048,
)

In [ ]:
out = llm.generate(
    ['The capital of France is'],
    SamplingParams(temperature=0, max_tokens=32)
)
print(out[0].outputs[0].text)

## 검증

위 `llm.generate(...)` 출력 위에 다음 두 줄이 찍혀야 한다:

```
(EngineCore pid=XXXXX) WARNING ... MyTritonImpl[v2/block_ptr].forward fired (prefill) tokens=5
(EngineCore pid=XXXXX) WARNING ... MyTritonImpl[v2/block_ptr].forward fired (decode, split-KV) s_len=6
```

`[v2/block_ptr]` 태그 — ptr 변형 (`[v2]`) 및 vllm_split (`fired`) 와 구분.

짧은 generation (max_tokens=32) 에선 대부분 step 의 KV 길이가 _SPLIT_KV_THRESHOLD (512) 미만 → wrapper 가 KV_SPLITS=1 fast path (block_ptr decode_simple) 로 분기. 긴 컨텍스트에서만 partial+reduce 두 커널 launch.

In [ ]:
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.CUSTOM.get_path()
print('CUSTOM slot ->', path)
assert 'MyTritonBackend' in path, 'CUSTOM slot 에 우리 백엔드가 등록되지 않음'
print('OK — 노트북 프로세스에서도 plugin 자동 등록 확인')
print()
print('실제 split-KV decode 실행 증거는 위 cell 11 출력 바로 위의 `[v2/block_ptr].forward fired (decode, split-KV)` 로그로 확인.')

## 다음 단계

- **paged KV 위 split-KV (`vllm_paged_v2`)** — 같은 split-KV idea 를 paged KV cache 와 결합. block_ptr idiom 에선 `BLOCK_N == BLOCK_SIZE` 강제 (BLOCK_PTR_MIGRATION.md §10) 와 reduction 호환성 별도 고려 필요
- **GQA head packing (`BLOCK_M = n_rep`)** — vLLM v1 production 의 추가 최적화. `tl.dot` 사용 가능 + KV 재사용
- **two 변형 `ptr/vllm_split_v2` 와 `block_ptr/vllm_split_v2` 의 PTX 비교** — 같은 split-KV 알고리즘이 두 메모리 접근 표현으로 어떻게 다른 명령으로 나오는지 확인